# Adult / Census Income EDA

## Problem Statement
Before building any machine learning model, we must first understand the dataset deeply.

This notebook performs **in-depth exploratory data analysis** on the Adult / Census Income dataset.

## What this notebook covers
- raw data loading
- missing value analysis
- duplicate checks
- numerical profiling with **NumPy**
- categorical profiling with **Pandas**
- univariate EDA
- target analysis
- bivariate analysis
- correlation analysis
- outlier detection
- probability checks
- confidence interval
- hypothesis testing
- cleaned CSV export
- GitHub-ready outputs

## Telugu
Model build cheyyadam mundu data ni deep ga ardham chesukovali.  
Ee notebook lo manam:
- data structure chustham
- missing values analyze chestham
- numerical and categorical columns separate chestham
- target distribution chustham
- feature vs income relations analyze chestham
- probability and statistics use chestham
- cleaned CSV save chestham

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
from math import sqrt

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from scipy.stats import chi2_contingency, ttest_ind

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

sns.set_theme(style="whitegrid")

os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../reports/figures", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

## Why these libraries?

### NumPy
Used for:
- fast numerical computation
- arrays
- percentiles
- variance
- standard deviation
- outlier bounds

### Pandas
Used for:
- loading CSV
- cleaning data
- grouping
- aggregation
- filtering
- saving cleaned CSV

### Matplotlib
Used for:
- basic plots
- histograms
- line plots
- figure saving

### Seaborn
Used for:
- statistical visualizations
- boxplots
- countplots
- heatmaps

### Plotly
Used for:
- interactive visualizations
- portfolio/demo quality charts

### Telugu
- NumPy = fast math engine
- Pandas = data handling king
- Matplotlib = plotting base
- Seaborn = beautiful EDA charts
- Plotly = interactive analytics

## Folder structure expected

```text
eda-adult-income/
├── data/
│   ├── raw/
│   │   └── adult.csv
│   └── processed/
├── notebooks/
│   └── adult_income_eda.ipynb
├── reports/
│   ├── figures/
│   └── insights.md
├── README.md
└── requirements.txt
```

In [ ]:
cols = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]

## Step 1: Load the official UCI raw file

Place the Adult dataset CSV inside:

`../data/raw/adult.csv`

This version assumes the raw file has **no header row**.

In [ ]:
df = pd.read_csv(
    "../data/raw/adult.csv",
    header=None,
    names=cols,
    na_values="?",
    skipinitialspace=True
)

print("Shape:", df.shape)
display(df.head())

In [ ]:
print(df.info())
print("\nDescriptive summary:")
display(df.describe(include="all").T)

## Why use `info()` and `describe()`?

- `info()` tells row count, dtypes, and missing hints.
- `describe()` gives summary statistics for both numerical and categorical columns.

### Telugu
Idi raw structure understanding ki first step.

In [ ]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

df["income"] = df["income"].str.replace(".", "", regex=False).str.strip()

display(df.head())

## Why clean string columns?
Strings lo spaces unte categories separate ga treat avuthayi.  
Example: `>50K` and ` >50K` different values laga behave avvachu.

In [ ]:
missing_report = (
    df.isna()
      .sum()
      .to_frame("missing_count")
      .assign(missing_pct=lambda x: (x["missing_count"] / len(df) * 100).round(2))
      .sort_values("missing_pct", ascending=False)
)

display(missing_report)
missing_report.to_csv("../data/processed/adult_missing_report.csv")

## Why missing value analysis?
Missing data random ga undochu, systematic ga undochu, or important signal kuda undochu.

### Telugu
Null ekkada undi ani first measure cheyyali.  
Direct ga fill cheyyakudadhu.

In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

In [ ]:
df = df.drop_duplicates().copy()
print("Shape after duplicate removal:", df.shape)

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

# NumPy-based Numerical Profiling

Here we directly use NumPy for:
- mean
- median
- std
- variance
- percentiles

Idi pure numerical understanding kosam useful.

In [ ]:
numpy_summary = []

for col in num_cols:
    arr = df[col].dropna().to_numpy()

    numpy_summary.append({
        "column": col,
        "count": arr.size,
        "mean": float(np.mean(arr)),
        "median": float(np.median(arr)),
        "std": float(np.std(arr)),
        "var": float(np.var(arr)),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "p01": float(np.percentile(arr, 1)),
        "p05": float(np.percentile(arr, 5)),
        "p25": float(np.percentile(arr, 25)),
        "p50": float(np.percentile(arr, 50)),
        "p75": float(np.percentile(arr, 75)),
        "p95": float(np.percentile(arr, 95)),
        "p99": float(np.percentile(arr, 99))
    })

numeric_summary_df = pd.DataFrame(numpy_summary).sort_values("column")
display(numeric_summary_df)
numeric_summary_df.to_csv("../data/processed/adult_numeric_summary.csv", index=False)

### Why mean, median, std, percentiles?
- **Mean** = average, but outlier-sensitive
- **Median** = robust center
- **Std / Variance** = spread
- **Percentiles** = distribution shape

In [ ]:
cat_summary = []

for col in cat_cols:
    mode_series = df[col].mode(dropna=True)
    cat_summary.append({
        "column": col,
        "n_unique": int(df[col].nunique(dropna=True)),
        "most_frequent": mode_series.iloc[0] if len(mode_series) > 0 else np.nan,
        "missing_count": int(df[col].isna().sum()),
        "missing_pct": round(df[col].isna().mean() * 100, 2)
    })

cat_summary_df = pd.DataFrame(cat_summary).sort_values("n_unique", ascending=False)
display(cat_summary_df)
cat_summary_df.to_csv("../data/processed/adult_categorical_summary.csv", index=False)

## Why categorical profiling?
Category columns lo:
- unique levels
- dominant levels
- missing percentages  
ivanni later encoding decisions ki important.

In [ ]:
df[num_cols].hist(figsize=(14, 10), bins=30)
plt.suptitle("Numerical Feature Distributions", y=1.02, fontsize=16)
plt.tight_layout()
plt.savefig("../reports/figures/numerical_histograms.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
for col in num_cols:
    plt.figure(figsize=(9, 4))
    sns.histplot(df[col], bins=30, kde=True)
    plt.title(f"Distribution of {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
for col in num_cols:
    plt.figure(figsize=(9, 3.5))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot - {col}")
    plt.tight_layout()
    plt.show()

## Why histograms and boxplots?
- histogram = shape, skew, concentration
- boxplot = median, IQR, outliers

In [ ]:
for col in cat_cols:
    print(f"\n===== {col} =====")
    display(df[col].value_counts(dropna=False).head(20))

In [ ]:
for col in cat_cols:
    plt.figure(figsize=(11, 5))
    order = df[col].value_counts(dropna=False).index[:15]
    sns.countplot(data=df, x=col, order=order)
    plt.xticks(rotation=45, ha="right")
    plt.title(f"Countplot - {col}")
    plt.tight_layout()
    plt.show()

## Why countplots?
Most frequent categories, imbalance, and rare values immediate ga telustayi.

In [ ]:
income_counts = df["income"].value_counts(dropna=False)
income_pct = (df["income"].value_counts(normalize=True, dropna=False) * 100).round(2)

print("Counts:")
display(income_counts.to_frame("count"))

print("Percentages:")
display(income_pct.to_frame("pct"))

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="income")
plt.title("Income Target Distribution")
plt.tight_layout()
plt.savefig("../reports/figures/income_target_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

## Why target analysis?
Classification model build cheyyadam mundu target balanced aa imbalance aa chudali.

In [ ]:
for col in num_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x="income", y=col)
    plt.title(f"{col} vs Income")
    plt.tight_layout()
    plt.show()

In [ ]:
group_stats = df.groupby("income")[num_cols].agg(["mean", "median", "std", "min", "max"])
display(group_stats)

## Why numeric vs target analysis?
High-income and low-income groups madhya actual numeric difference ento chudataniki.

In [ ]:
important_cat_cols = [c for c in ["education", "workclass", "marital_status", "occupation", "relationship", "sex", "race", "native_country"] if c in df.columns]

for col in important_cat_cols:
    ct = pd.crosstab(df[col], df["income"], normalize="index").mul(100).round(2)
    print(f"\n===== {col} vs income (%) =====")
    display(ct.head(20))

In [ ]:
for col in important_cat_cols:
    temp = pd.crosstab(df[col], df["income"], normalize="index")
    temp.plot(kind="bar", stacked=True, figsize=(11, 5))
    plt.title(f"{col} vs Income Proportion")
    plt.ylabel("Proportion")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
corr = df[num_cols].corr(numeric_only=True)
display(corr)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap - Numerical Features")
plt.tight_layout()
plt.savefig("../reports/figures/correlation_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

## Why correlation?
Useful for numeric feature relationships, but remember: **correlation ≠ causation**

In [ ]:
skew_report = df[num_cols].skew().sort_values(ascending=False).to_frame("skewness")
display(skew_report)
skew_report.to_csv("../data/processed/adult_skew_report.csv")

In [ ]:
outlier_rows = []

for col in num_cols:
    arr = df[col].dropna().to_numpy()

    q1 = np.percentile(arr, 25)
    q3 = np.percentile(arr, 75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_count = int(((arr < lower) | (arr > upper)).sum())
    outlier_pct = round((outlier_count / len(arr)) * 100, 2)

    outlier_rows.append({
        "column": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": outlier_count,
        "outlier_pct": outlier_pct
    })

outlier_report = pd.DataFrame(outlier_rows).sort_values("outlier_pct", ascending=False)
display(outlier_report)
outlier_report.to_csv("../data/processed/adult_outlier_report.csv", index=False)

## Why outlier detection?
Outliers mean ni distort chestayi, variance ni inflate chestayi, and some models meeda impact untundi.  
But blind ga remove cheyyakudadhu.

In [ ]:
p_high_income = (df["income"] == ">50K").mean()
print("P(income > 50K) =", round(p_high_income, 4))

In [ ]:
prob_results = {}

if "education" in df.columns:
    for edu in ["Bachelors", "Masters", "Doctorate", "HS-grad"]:
        temp = df.loc[df["education"] == edu, "income"]
        if len(temp) > 0:
            prob_results[f"P(>50K | education={edu})"] = round(temp.eq(">50K").mean(), 4)

if "sex" in df.columns:
    for s in df["sex"].dropna().unique():
        temp = df.loc[df["sex"] == s, "income"]
        prob_results[f"P(>50K | sex={s})"] = round(temp.eq(">50K").mean(), 4)

display(pd.Series(prob_results, name="probability").to_frame())

## Why probability?
Counts kanna probability better understanding isthundi:
- base rate
- conditional rate

In [ ]:
n = df["income"].notna().sum()
p = (df["income"] == ">50K").mean()
z = 1.96

margin = z * sqrt((p * (1 - p)) / n)
lower = p - margin
upper = p + margin

print("Estimated proportion of >50K:", round(p, 4))
print("95% Confidence Interval:", (round(lower, 4), round(upper, 4)))

## Why confidence interval?
Single estimate kanna uncertainty range honest ga untundi.

In [ ]:
age_high = df.loc[df["income"] == ">50K", "age"].dropna()
age_low = df.loc[df["income"] == "<=50K", "age"].dropna()

t_stat, p_value = ttest_ind(age_high, age_low, equal_var=False)

print("Age difference t-test")
print("T-statistic:", t_stat)
print("P-value:", p_value)

In [ ]:
cont_table = pd.crosstab(df["education"], df["income"])
chi2, p_chi, dof, expected = chi2_contingency(cont_table)

print("Education vs Income chi-square test")
print("Chi-square statistic:", chi2)
print("P-value:", p_chi)
print("Degrees of freedom:", dof)

## Why hypothesis testing?
Visual difference kanipinchindi ante actual signal aa leka random aa ani verify cheyyali.

In [ ]:
income_by_education = (
    df.assign(high_income=lambda x: (x["income"] == ">50K").astype(int))
      .groupby("education", as_index=False)
      .agg(
          total_people=("income", "size"),
          high_income_count=("high_income", "sum"),
          high_income_rate=("high_income", "mean")
      )
      .sort_values("high_income_rate", ascending=False)
)

income_by_education["high_income_rate"] = (income_by_education["high_income_rate"] * 100).round(2)
display(income_by_education.head(15))
income_by_education.to_csv("../data/processed/income_by_education.csv", index=False)

In [ ]:
income_by_workclass = (
    df.assign(high_income=lambda x: (x["income"] == ">50K").astype(int))
      .groupby("workclass", as_index=False)
      .agg(
          total_people=("income", "size"),
          high_income_count=("high_income", "sum"),
          high_income_rate=("high_income", "mean")
      )
      .sort_values("high_income_rate", ascending=False)
)

income_by_workclass["high_income_rate"] = (income_by_workclass["high_income_rate"] * 100).round(2)
display(income_by_workclass.head(15))
income_by_workclass.to_csv("../data/processed/income_by_workclass.csv", index=False)

In [ ]:
sex_income_summary = pd.crosstab(df["sex"], df["income"], margins=True)
display(sex_income_summary)
sex_income_summary.to_csv("../data/processed/sex_income_summary.csv")

## Why grouped summaries?
Professional analytics work lo only charts kaadu, grouped tables kuda chala important.

In [ ]:
fig = px.histogram(
    df,
    x="age",
    color="income",
    barmode="overlay",
    nbins=40,
    marginal="box",
    title="Interactive Age Distribution by Income"
)
fig.write_html("../reports/figures/interactive_age_distribution.html")
fig.show()

In [ ]:
fig = px.box(
    df,
    x="income",
    y="hours_per_week",
    color="income",
    title="Interactive Hours per Week by Income"
)
fig.write_html("../reports/figures/interactive_hours_vs_income.html")
fig.show()

In [ ]:
edu_plot_df = (
    df.assign(high_income=lambda x: (x["income"] == ">50K").astype(int))
      .groupby("education", as_index=False)["high_income"]
      .mean()
      .sort_values("high_income", ascending=False)
)

edu_plot_df["high_income"] = (edu_plot_df["high_income"] * 100).round(2)

fig = px.bar(
    edu_plot_df,
    x="education",
    y="high_income",
    title="Percentage of >50K by Education"
)
fig.update_layout(xaxis_tickangle=-45)
fig.write_html("../reports/figures/interactive_education_income.html")
fig.show()

## Why Plotly?
GitHub portfolio and demo videos ki interactive visuals strong ga untayi.

In [ ]:
df_clean = df.copy()

for col in ["workclass", "occupation", "native_country"]:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("Unknown")

for col in df_clean.select_dtypes(include="object").columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()

print("Missing values after cleaning:")
display(df_clean.isna().sum().to_frame("missing_count"))

## Why fill categorical missing with `Unknown`?
Rows direct ga drop cheyyadam kanna explicit unknown category better first-pass strategy.

In [ ]:
df_clean.to_csv("../data/processed/adult_cleaned.csv", index=False)
print("Saved cleaned CSV: ../data/processed/adult_cleaned.csv")

In [ ]:
insights = '''
# Adult Income EDA Insights

1. The income target is imbalanced, with the <=50K class appearing more frequently than >50K.
2. Numerical features such as age and hours_per_week show visible differences across income groups.
3. Capital_gain and capital_loss are highly skewed and may require careful treatment before modeling.
4. Education appears strongly associated with higher-income outcomes.
5. Workclass, occupation, and native_country contain missing-like values and required cleaning.
6. Some numerical features contain outliers or long tails, especially in financial-style columns.
7. Category-level income proportions provide more useful insight than raw counts alone.
8. A cleaned dataset was created and saved for downstream feature engineering and model building.
'''

with open("../reports/insights.md", "w", encoding="utf-8") as f:
    f.write(insights)

print(insights)

# Final outputs created

## data/processed/
- adult_cleaned.csv
- adult_missing_report.csv
- adult_numeric_summary.csv
- adult_categorical_summary.csv
- adult_outlier_report.csv
- adult_skew_report.csv
- income_by_education.csv
- income_by_workclass.csv
- sex_income_summary.csv

## reports/figures/
- numerical_histograms.png
- income_target_distribution.png
- correlation_heatmap.png
- interactive_age_distribution.html
- interactive_hours_vs_income.html
- interactive_education_income.html

## reports/
- insights.md